## 1. Authenticate with NASA Earthdata

In [6]:
import earthaccess

auth = earthaccess.login()
print("Authenticated:", auth.authenticated)

/home/aatraya/SIH_2026/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authenticated: True


## 2. Load the DEM and compute slope

In [7]:
import rasterio
import numpy as np

dem_path = "Sikkim.tif"

with rasterio.open(dem_path) as dem:
    dem_array = dem.read(1).astype(np.float32)
    dem_transform = dem.transform
    dem_crs = dem.crs
    dem_bounds = dem.bounds
    dem_nodata = dem.nodata

# Mask nodata values
dem_array = np.where(dem_array == dem_nodata, np.nan, dem_array)

print("Shape:", dem_array.shape)
print("Bounds:", dem_bounds)
print("CRS:", dem_crs)
print("Nodata value:", dem_nodata)
print("Elevation range:", np.nanmin(dem_array), "to", np.nanmax(dem_array))

Shape: (3285, 2294)
Bounds: BoundingBox(left=88.1815277778133, bottom=27.00486111110674, right=88.8187500000356, top=27.917361111106864)
CRS: EPSG:4326
Nodata value: -32768.0
Elevation range: 183.0 to 7346.0


In [8]:
from scipy import ndimage

# Approx pixel size in meters (SRTM is in degrees -> convert using Sikkim's latitude)
pixel_size_x = dem_transform[0] * 111320 * np.cos(np.radians(27.5))
pixel_size_y = abs(dem_transform[4]) * 111320

dzdx = ndimage.sobel(dem_array, axis=1) / (8 * pixel_size_x)
dzdy = ndimage.sobel(dem_array, axis=0) / (8 * pixel_size_y)

slope_rad = np.arctan(np.sqrt(dzdx**2 + dzdy**2))
slope_deg = np.degrees(slope_rad)

print("Slope range:", np.nanmin(slope_deg), "to", np.nanmax(slope_deg))
print("Slope 95th percentile:", np.nanpercentile(slope_deg, 95))
print("Slope mean:", np.nanmean(slope_deg))

Slope range: 0.0 to 86.72215997032231
Slope 95th percentile: 49.68170839008894
Slope mean: 28.95248176620194


## 3. SMAP soil moisture data

Quick sanity check that SMAP granules are available for the AOI/date range via `earthaccess`
(the bulk 2022-2024 series used below was pulled separately through an AppEEARS request, since
that gives clipped, ready-to-use GeoTIFFs instead of raw granules).

In [9]:
results = earthaccess.search_data(
    short_name="SPL3SMP_E",
    bounding_box=(dem_bounds.left, dem_bounds.bottom, dem_bounds.right, dem_bounds.top),
    temporal=("2024-06-01", "2024-06-10")
)
print("Granules found:", len(results))

Granules found: 10


In [10]:
import glob

sm_files = sorted(glob.glob("smap_data/*soil_moisture*.tif"))
print("Total soil moisture files:", len(sm_files))
print("First file:", sm_files[0])

with rasterio.open(sm_files[0]) as src:
    sm_array = src.read(1)
    sm_bounds = src.bounds
    sm_crs = src.crs
    sm_nodata = src.nodata

sm_array = np.where(sm_array == sm_nodata, np.nan, sm_array)

print("Shape:", sm_array.shape)
print("Bounds:", sm_bounds)
print("CRS:", sm_crs)
print("Nodata:", sm_nodata)
print("Value range:", np.nanmin(sm_array), "to", np.nanmax(sm_array))

Total soil moisture files: 383
First file: smap_data/SPL3SMP_E.006_Soil_Moisture_Retrieval_Data_AM_soil_moisture_20220101T000000_aid0001.tif
Shape: (12, 9)
Bounds: BoundingBox(left=88.15610219843427, bottom=26.97846511923281, right=88.87023803982572, top=27.930646241088084)
CRS: EPSG:4326
Nodata: -9999.0
Value range: 0.1273873 to 0.36533302


## 4. Pull the full soil-moisture bundle from AppEEARS

In [1]:
import netrc
import requests

# Read credentials from .netrc automatically
secrets = netrc.netrc()
username, _, password = secrets.authenticators("urs.earthdata.nasa.gov")

login_resp = requests.post(
    "https://appeears.earthdatacloud.nasa.gov/api/login",
    auth=(username, password)
)
login_resp.raise_for_status()
token = login_resp.json()["token"]
print("Login status:", login_resp.status_code)

Login status: 200


In [2]:
headers = {"Authorization": f"Bearer {token}"}
task_id = "4247fc3c-ee4b-4e75-95b3-c59354eafc9c"
bundle_url = f"https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}"

response = requests.get(bundle_url, headers=headers)
response.raise_for_status()
bundle = response.json()
files = bundle["files"]
print("Total files in bundle:", len(files))

Total files in bundle: 1440


In [3]:
soil_moisture_files = [f for f in files if "soil_moisture" in f["file_name"] and "qual_flag" not in f["file_name"]]
qual_flag_files = [f for f in files if "qual_flag" in f["file_name"]]
other_files = [f for f in files if "soil_moisture" not in f["file_name"] and "qual_flag" not in f["file_name"]]

print("Soil moisture files:", len(soil_moisture_files))
print("Qual flag files:", len(qual_flag_files))
print("Other/supporting files:", len(other_files))

Soil moisture files: 383
Qual flag files: 1050
Other/supporting files: 7


In [4]:
import re

dates = sorted(re.search(r"(\d{8})T000000", f["file_name"]).group(1) for f in soil_moisture_files)
print("Earliest:", dates[0])
print("Latest:", dates[-1])
print("Total unique dates:", len(set(dates)))

Earliest: 20220101
Latest: 20241230
Total unique dates: 383


## 5. Load and reproject soil moisture onto the DEM grid

In [11]:
import os

def load_soil_moisture(filepath):
    with rasterio.open(filepath) as src:
        arr = src.read(1)
        nodata = src.nodata
        arr = np.where(arr == nodata, np.nan, arr)
    return arr

# Quick test on the first file
filename = os.path.basename(soil_moisture_files[0]["file_name"])
local_path = os.path.join("smap_data", filename)

test_arr = load_soil_moisture(local_path)
print(test_arr.shape, np.nanmin(test_arr), np.nanmax(test_arr))

(12, 9) 0.1273873 0.36533302


In [12]:
from rasterio.warp import reproject, Resampling

def resample_soil_moisture(filepath, dem_array, dem_transform, dem_crs):
    with rasterio.open(filepath) as sm_src:
        sm_array = sm_src.read(1)
        sm_transform = sm_src.transform
        sm_crs = sm_src.crs
        sm_nodata = sm_src.nodata

    sm_array = np.where(sm_array == sm_nodata, np.nan, sm_array)

    sm_resampled = np.empty(dem_array.shape, dtype=np.float32)
    reproject(
        source=sm_array,
        destination=sm_resampled,
        src_transform=sm_transform,
        src_crs=sm_crs,
        dst_transform=dem_transform,
        dst_crs=dem_crs,
        resampling=Resampling.bilinear
    )
    return sm_resampled

# Sanity check on the same test file
sm_resampled = resample_soil_moisture(local_path, dem_array, dem_transform, dem_crs)
print("Resampled shape:", sm_resampled.shape, "| DEM shape:", dem_array.shape)
print("Value range after resample:", np.nanmin(sm_resampled), np.nanmax(sm_resampled))

Resampled shape: (3285, 2294) | DEM shape: (3285, 2294)
Value range after resample: 0.13012834 0.36530203


In [13]:
soil_moisture_stack = {}

for f in soil_moisture_files:
    filename = os.path.basename(f["file_name"])
    date_str = re.search(r"(\d{8})T000000", filename).group(1)
    local_path = os.path.join("smap_data", filename)

    soil_moisture_stack[date_str] = resample_soil_moisture(local_path, dem_array, dem_transform, dem_crs)

print("Total dates processed:", len(soil_moisture_stack))

Total dates processed: 383


## 6. Landslide points

Start from the known, high-confidence October 2023 events, then try to enrich them with NASA's
Global Landslide Catalog (a one-time export, current as of March 2016 -- so it won't contain the
2023 Sikkim events, but can add other historical points inside the AOI). Falls back to the manual
list alone if the download fails.

In [14]:
import pandas as pd

manual_landslides = pd.DataFrame([
    {"latitude": 27.6022, "longitude": 88.6463, "date": "20231004", "location": "Chungthang Dam", "label": 1},
    {"latitude": 27.4975, "longitude": 88.5348, "date": "20231004", "location": "Mangan", "label": 1},
    {"latitude": 27.4005, "longitude": 88.5135, "date": "20231004", "location": "Dikchu", "label": 1},
    {"latitude": 27.2341, "longitude": 88.4977, "date": "20231004", "location": "Singtam", "label": 1},
    {"latitude": 27.1751, "longitude": 88.5333, "date": "20231004", "location": "Rangpo", "label": 1},
])

glc_url = "https://data.nasa.gov/docs/legacy/Global_Landslide_Catalog_Export/Global_Landslide_Catalog_Export_rows.csv"

try:
    df_glc = pd.read_csv(glc_url)

    sikkim_glc = df_glc[
        (df_glc["latitude"] >= dem_bounds.bottom) & (df_glc["latitude"] <= dem_bounds.top) &
        (df_glc["longitude"] >= dem_bounds.left) & (df_glc["longitude"] <= dem_bounds.right)
    ].copy()

    if not sikkim_glc.empty:
        sikkim_glc["date"] = pd.to_datetime(sikkim_glc["event_date"]).dt.strftime("%Y%m%d")
        sikkim_glc["location"] = sikkim_glc["location_description"].fillna("NASA_GLC")
        sikkim_glc["label"] = 1
        sikkim_glc = sikkim_glc[["latitude", "longitude", "date", "location", "label"]]

        df_points = pd.concat([manual_landslides, sikkim_glc], ignore_index=True)
        print(f"Added {len(sikkim_glc)} points from the NASA Global Landslide Catalog.")
    else:
        print("No GLC points fall inside the Sikkim bounding box. Using manual points only.")
        df_points = manual_landslides

except Exception as e:
    print(f"GLC download failed ({e}). Falling back to manual points only.")
    df_points = manual_landslides

print(f"\nTotal positive landslide points: {len(df_points)}")
print(df_points)

Added 78 points from the NASA Global Landslide Catalog.

Total positive landslide points: 83
     latitude  longitude      date                                  location  \
0   27.602200  88.646300  20231004                            Chungthang Dam   
1   27.497500  88.534800  20231004                                    Mangan   
2   27.400500  88.513500  20231004                                    Dikchu   
3   27.234100  88.497700  20231004                                   Singtam   
4   27.175100  88.533300  20231004                                    Rangpo   
..        ...        ...       ...                                       ...   
78  27.604700  88.646300  20130908                        Chungthang, Sikkim   
79  27.267600  88.289100  20161012                                   Legship   
80  27.077100  88.482000  20080630                                 Kalimpong   
81  27.043571  88.265221  20090526  Darjeeling Sadar, Darjeeling West Bengal   
82  27.360364  88.651914  2

## 7. Extract DEM / slope / soil-moisture features at each landslide point

In [15]:
def extract_point_features(points_df, dem_transform, dem_array, slope_array, smap_stack):
    dataset = []

    for _, row in points_df.iterrows():
        lat, lon = row["latitude"], row["longitude"]
        event_date = str(row["date"])

        # Convert geographic lat/lon to array row, col indices
        col, r = ~dem_transform * (lon, lat)
        row_idx, col_idx = int(r), int(col)

        if 0 <= row_idx < dem_array.shape[0] and 0 <= col_idx < dem_array.shape[1]:
            elevation = dem_array[row_idx, col_idx]
            slope = slope_array[row_idx, col_idx]
            sm_val = smap_stack[event_date][row_idx, col_idx] if event_date in smap_stack else np.nan

            dataset.append({
                "latitude": lat,
                "longitude": lon,
                "elevation": elevation,
                "slope": slope,
                "soil_moisture": sm_val,
                "label": row["label"],
            })

    return pd.DataFrame(dataset)

positive_samples = extract_point_features(df_points, dem_transform, dem_array, slope_deg, soil_moisture_stack)
print(positive_samples)

     latitude  longitude  elevation      slope  soil_moisture  label
0   27.602200  88.646300     1578.0   8.317919            NaN      1
1   27.497500  88.534800     1193.0  20.508342            NaN      1
2   27.400500  88.513500      894.0  53.652324            NaN      1
3   27.234100  88.497700      458.0  29.361959            NaN      1
4   27.175100  88.533300      302.0   1.046990            NaN      1
..        ...        ...        ...        ...            ...    ...
78  27.604700  88.646300     1614.0  13.507316            NaN      1
79  27.267600  88.289100      614.0  49.122081            NaN      1
80  27.077100  88.482000     1222.0  17.086431            NaN      1
81  27.043571  88.265221     2070.0   2.966469            NaN      1
82  27.360364  88.651914     2250.0  44.829349            NaN      1

[83 rows x 6 columns]
